## 4.1 Setup

In [ ]:
import os

REPO_URL = "https://github.com/meriem200512365/Chat-boot-cegedim.git"
REPO_DIR = "Chat-boot-cegedim"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}


## 4.2 Chargement du rapport d'anomalies

In [ ]:
import json

with open("data/chemins/liens_casses.json", encoding="utf-8") as f:
    rapport = json.load(f)

print(json.dumps(rapport, indent=2, ensure_ascii=False))


## 4.3 Vue d'ensemble

In [ ]:
import pandas as pd

resume = pd.Series({
    "Menus totaux": rapport["nb_menus_total"],
    "Chemins valides generes": rapport["nb_chemins_valides"],
    "Liens casses": rapport["nb_liens_casses"],
    "Menus orphelins": rapport["nb_menus_orphelins"],
})
resume


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
resume.plot(kind="bar", color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])
plt.title("Etat du menu.xml apres extraction")
plt.ylabel("Nombre")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 4.4 Détail des liens cassés (si présents)

Chaque entrée montre le menu et l'item fautif, ainsi que le nom du
sous-menu attendu mais introuvable.

In [ ]:
if rapport["liens_casses"]:
    df_casses = pd.DataFrame(rapport["liens_casses"])
    display(df_casses)

    print("\nRepartition des liens casses par menu parent :")
    print(df_casses["menu_parent"].value_counts() if "menu_parent" in df_casses.columns else "colonne non presente, voir cles disponibles :", df_casses.columns.tolist())
else:
    print("Aucun lien casse detecte sur cette version de menu.xml (etat propre).")


## 4.5 Détail des menus orphelins (si présents)

In [ ]:
if rapport["menus_orphelins"]:
    for m in rapport["menus_orphelins"]:
        print(" -", m)
else:
    print("Aucun menu orphelin detecte sur cette version de menu.xml.")


## 4.6 Suivi dans le temps (régression)

`menu.xml` est amené à évoluer (nouvelles fonctionnalités, réorganisation
du menu côté Activ'Premium). Cette cellule permet de **regénérer le
rapport à la volée** depuis le XML actuel et de le comparer à celui
versionné dans le dépôt, pour détecter une régression avant de réindexer
en production.

In [ ]:
import sys
sys.path.insert(0, ".")

from src.extraction.xml_parser import parse_menu_xml
from src.extraction.path_builder import build_paths

main_menu, menus = parse_menu_xml()
chemins, liens_casses_actuels, orphelins_actuels = build_paths(main_menu, menus)

print(f"Chemins valides (recalcules) : {len(chemins)}  (rapport versionne : {rapport['nb_chemins_valides']})")
print(f"Liens casses (recalcules)    : {len(liens_casses_actuels)}  (rapport versionne : {rapport['nb_liens_casses']})")
print(f"Menus orphelins (recalcules) : {len(orphelins_actuels)}  (rapport versionne : {rapport['nb_menus_orphelins']})")

if len(liens_casses_actuels) != rapport["nb_liens_casses"] or len(orphelins_actuels) != rapport["nb_menus_orphelins"]:
    print("\n[ATTENTION] Le menu.xml actuel differe du rapport versionne -> relancer scripts/update_index.py")
else:
    print("\nOK : coherent avec le rapport versionne dans le depot.")
